<a href="https://colab.research.google.com/github/NelvaAdalit/-INTELIGENCIA-ARTIFICIAL-I-/blob/main/KMEANSGROUPIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# 1. Conectar con Google Drive
from google.colab import drive
drive.mount('/content/gdrive')

# 2. Importar las librerías necesarias
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore') # Evita advertencias rojas en la consola

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [7]:
# Definir la ruta exacta que proporcionaste
ruta_dataset = '/content/gdrive/MyDrive/SIS420IA/dataset.csv'
# Cargar los datos
df = pd.read_csv(ruta_dataset)

# Mostrar información fundamental (filas, columnas y tipos de datos)
print("--- INFORMACIÓN DEL DATASET ---")
print(f"Total de Filas (m): {df.shape[0]}")
print(f"Total de Columnas (n): {df.shape[1]}")
print(f"Total de Celdas: {df.size}")
print("-" * 30)

# Mostrar las primeras 5 filas para verificar que cargó bien
df.head()

--- INFORMACIÓN DEL DATASET ---
Total de Filas (m): 114000
Total de Columnas (n): 21
Total de Celdas: 2394000
------------------------------


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [8]:
# Quedarnos solo con características numéricas y quitar nulos
df_num = df.select_dtypes(include=[np.number]).dropna()

print(f"Datos tras limpieza listos para clustering: {df_num.shape[0]} filas y {df_num.shape[1]} columnas.")

# Aplicar escalado estándar (Media 0, Varianza 1)
scaler = StandardScaler()
datos_escalados = scaler.fit_transform(df_num)

# Guardar en un DataFrame
df_escalado = pd.DataFrame(datos_escalados, columns=df_num.columns)
print("¡Escalado completado exitosamente!")

Datos tras limpieza listos para clustering: 114000 filas y 15 columnas.
¡Escalado completado exitosamente!


In [9]:
# Aplicar PCA para 3 dimensiones
pca = PCA(n_components=3)
datos_pca = pca.fit_transform(df_escalado)

# Crear dataset para la gráfica 3D
df_pca_3d = pd.DataFrame(datos_pca, columns=['Componente_1', 'Componente_2', 'Componente_3'])

# Varianza explicada (Qué tanta información conservamos)
var_explicada = pca.explained_variance_ratio_.sum() * 100
print(f"Los 3 componentes principales capturan el {var_explicada:.2f}% de la varianza total de tus {df_num.shape[1]} columnas originales.")

Los 3 componentes principales capturan el 38.62% de la varianza total de tus 15 columnas originales.


In [11]:
from sklearn.cluster import MiniBatchKMeans
import plotly.graph_objects as go

valores_k = [3, 5, 7, 9]

for k in valores_k:
    print(f"\n{'='*50}")
    print(f"ENTRENANDO MINIBATCH K-MEANS CON K = {k}")
    print(f"{'='*50}")

    # 1. Instanciar MiniBatchKMeans (Aquí está el bache a bache, procesando de a 2048 datos a la vez)
    minibatch_kmeans = MiniBatchKMeans(
        n_clusters=k,
        batch_size=2048, # Tamaño del mini-bache
        random_state=42,
        n_init='auto'
    )

    # Entrenar y predecir directamente con nuestro dataset escalado
    etiquetas = minibatch_kmeans.fit_predict(df_escalado)

    # 2. Guardar etiquetas en el dataset 3D (asumiendo que ya corriste la celda del PCA)
    df_pca_3d[f'Cluster_{k}'] = etiquetas.astype(str)

    # 3. Calcular el posicionamiento de los centroides
    centroides_originales = minibatch_kmeans.cluster_centers_
    centroides_3d = pca.transform(centroides_originales)

    # --- MUESTRA PARA GRAFICAR ---
    # Sacamos una muestra aleatoria para no congelar tu navegador con 114,000 puntos en 3D
    if df_pca_3d.shape[0] > 10000:
        df_plot = df_pca_3d.sample(n=10000, random_state=42)
    else:
        df_plot = df_pca_3d

    # 4. Construir la gráfica interactiva 3D
    fig = go.Figure()

    for i in range(k):
        datos_cluster = df_plot[df_plot[f'Cluster_{k}'] == str(i)]
        fig.add_trace(go.Scatter3d(
            x=datos_cluster['Componente_1'],
            y=datos_cluster['Componente_2'],
            z=datos_cluster['Componente_3'],
            mode='markers',
            marker=dict(size=3, opacity=0.5),
            name=f'Cluster {i}'
        ))

    # Añadir los Centroides (Diamantes)
    fig.add_trace(go.Scatter3d(
        x=centroides_3d[:, 0],
        y=centroides_3d[:, 1],
        z=centroides_3d[:, 2],
        mode='markers',
        marker=dict(size=12, color='black', symbol='diamond', line=dict(width=2, color='white')),
        name='Centroides'
    ))

    fig.update_layout(
        title=f'Dispersión 3D - MiniBatch KMeans (K={k})',
        scene=dict(xaxis_title='PCA 1', yaxis_title='PCA 2', zaxis_title='PCA 3'),
        width=900, height=700
    )

    fig.show()


ENTRENANDO MINIBATCH K-MEANS CON K = 3



ENTRENANDO MINIBATCH K-MEANS CON K = 5



ENTRENANDO MINIBATCH K-MEANS CON K = 7



ENTRENANDO MINIBATCH K-MEANS CON K = 9
